# 01 - Data Cleaning Sentiment Map RS Hermina

Notebook ini menjadi **file utama** untuk tahap cleaning data. Semua proses cleaning ditulis langsung di notebook supaya alurnya mudah dibaca dan tidak perlu berpindah ke file `.py`.

Tujuan cleaning sesuai overview project:
- menyiapkan teks ulasan bersih,
- membuat label awal sentimen positif/netral/negatif,
- mendeteksi aspek pelayanan,
- menyiapkan ringkasan cabang untuk sentiment map,
- menyiapkan data untuk cabang terbaik, cabang prioritas, dan masalah dominan.

## 1. Import Library dan Path

In [1]:
from pathlib import Path
import json
import re
import unicodedata
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT_PATH = ROOT / 'data_fix.csv'
DATA_DIR = ROOT / 'data'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = ROOT / 'reports'

for directory in [INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR, REPORTS_DIR / 'removed_rows']:
    directory.mkdir(parents=True, exist_ok=True)

INPUT_PATH

WindowsPath('d:/Scraper adnan/preprocessing/data_fix.csv')

## 2. Load Dataset Awal

Dataset awal berisi ulasan RS Hermina. Kolom pentingnya adalah cabang, provinsi, rating, waktu ulasan, dan teks ulasan.

In [2]:
raw = pd.read_csv(INPUT_PATH)
print('Shape:', raw.shape)
display(raw.head())
display(raw.isna().sum().to_frame('missing_value'))
display(raw.dtypes.to_frame('dtype'))

Shape: (14785, 7)


,review_id,official_name,official_province,google_name,review_rating,review_time_text,review_text
0,06b6d8e308a4f4ae8c2e,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,5.0,10 bulan lalu,Saya pasien kista kamar exs 618 alhamdulillah ...
1,92fa6935a492adb7b945,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,5.0,10 bulan lalu,Oprasi sc eracs sma dr.fardani mantap + super ...
2,03b141ff1a9b60082925,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,5.0,2 bulan lalu,Bismillahhorohmanirohim Saya perwakilan dari k...
3,04d83965b8830af12922,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,5.0,2 bulan lalu,Layanan Perinatologinya ramah dan bagus ☺️ ter...
4,0589e33a19a173dcf910,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,5.0,2 bulan lalu,"Pelayanan dari mulai penanganan IGD, perawatan..."


,missing_value
review_id,0
official_name,0
official_province,0
google_name,0
review_rating,0
review_time_text,566
review_text,0


,dtype
review_id,object
official_name,object
official_province,object
google_name,object
review_rating,float64
review_time_text,object
review_text,object


## 3. Konfigurasi Cleaning

Bagian ini berisi kamus normalisasi kata tidak baku dan daftar kata kunci aspek pelayanan.

In [3]:
REQUIRED_COLUMNS = [
    'review_id', 'official_name', 'official_province', 'google_name',
    'review_rating', 'review_time_text', 'review_text'
]

SLANG_MAP = {
    'gak': 'tidak', 'ga': 'tidak', 'ngga': 'tidak', 'nggak': 'tidak', 'tdk': 'tidak',
    'yg': 'yang', 'dgn': 'dengan', 'krn': 'karena', 'karna': 'karena',
    'bgt': 'banget', 'bangett': 'banget', 'sma': 'sama', 'sm': 'sama',
    'utk': 'untuk', 'tp': 'tapi', 'tpi': 'tapi', 'kalo': 'kalau',
    'trims': 'terima kasih', 'makasih': 'terima kasih', 'mksh': 'terima kasih',
    'gercep': 'gerak cepat', 'satset': 'cepat', 'cs': 'customer service'
}

ASPECT_KEYWORDS = {
    'antrean': ['antri', 'antrian', 'antre', 'antrean', 'menunggu', 'nunggu', 'tunggu', 'lama', 'waktu tunggu'],
    'dokter': ['dokter', 'dr.', 'dr ', 'dokternya', 'spesialis'],
    'perawat': ['perawat', 'suster', 'nurse', 'bidan'],
    'administrasi': ['administrasi', 'admin', 'pendaftaran', 'loket', 'receptionist', 'resepsionis', 'customer service', 'cs'],
    'farmasi': ['farmasi', 'obat', 'apotik', 'apotek', 'resep'],
    'fasilitas': ['fasilitas', 'ruang', 'kamar', 'toilet', 'parkir', 'ac', 'kursi', 'gedung', 'lift'],
    'biaya_bpjs': ['biaya', 'mahal', 'bayar', 'pembayaran', 'bpjs', 'asuransi', 'tagihan'],
    'kebersihan': ['bersih', 'kotor', 'kebersihan', 'bau', 'rapi', 'nyaman'],
    'komunikasi': ['ramah', 'jutek', 'jelas', 'informasi', 'respon', 'respons', 'komunikasi', 'sopan', 'helpful']
}

def normalize_space(value):
    return re.sub(r'\s+', ' ', value).strip()

def normalize_unicode(value):
    return unicodedata.normalize('NFKC', value)

def clean_text(value):
    text = '' if pd.isna(value) else str(value)
    text = normalize_unicode(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', ' ', text)
    text = re.sub(r'[^0-9a-zA-ZÀ-ÿ\s]', ' ', text)
    text = normalize_space(text)
    tokens = []
    for token in text.split():
        token = re.sub(r'([a-z])\1{2,}', r'\1\1', token)
        tokens.append(SLANG_MAP.get(token, token))
    return normalize_space(' '.join(tokens))

def sentiment_from_rating(rating):
    if rating <= 2:
        return 'negatif'
    if rating == 3:
        return 'netral'
    return 'positif'

def contains_keyword(text, keywords):
    padded = f' {text} '
    return any(keyword in padded for keyword in keywords)

def pct(part, whole):
    return round(part / whole * 100, 2) if whole else 0.0

def confidence_from_count(count):
    if count >= 200:
        return 'tinggi'
    if count >= 100:
        return 'sedang'
    return 'rendah'

## 4. Cleaning Data

Tahap ini membersihkan kolom teks, mengubah rating menjadi integer, menghapus duplikat, membuat label sentimen, dan membuat flag aspek pelayanan.

In [4]:
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(raw.columns))
if missing_columns:
    raise ValueError(f'Kolom wajib tidak ditemukan: {missing_columns}')

df = raw[REQUIRED_COLUMNS].copy()
input_rows = len(df)

for column in ['review_id', 'official_name', 'official_province', 'google_name', 'review_time_text', 'review_text']:
    df[column] = df[column].fillna('').astype(str).map(normalize_unicode).map(normalize_space)

df['official_name'] = df['official_name'].str.replace(r'\s+', ' ', regex=True).str.strip()
df['official_province'] = df['official_province'].str.upper().str.replace(r'\s+', ' ', regex=True).str.strip()
df['google_name'] = df['google_name'].str.replace(r'\s+', ' ', regex=True).str.strip()

df['review_rating_int'] = pd.to_numeric(df['review_rating'], errors='coerce').round().astype('Int64')
invalid_rating = df[df['review_rating_int'].isna() | ~df['review_rating_int'].between(1, 5)].copy()
df = df[df['review_rating_int'].between(1, 5)].copy()
df['review_rating_int'] = df['review_rating_int'].astype(int)

df['clean_text'] = df['review_text'].map(clean_text)
df['char_count'] = df['clean_text'].str.len()
df['word_count'] = df['clean_text'].str.split().str.len()
df['is_short_review'] = df['word_count'] < 3
df['sentiment_label'] = df['review_rating_int'].map(sentiment_from_rating)
df['sentiment_id'] = df['sentiment_label'].map({'negatif': 0, 'netral': 1, 'positif': 2})

before_review_id_dedup = len(df)
duplicate_review_id = df[df.duplicated('review_id', keep=False)].copy()
df = df.drop_duplicates('review_id', keep='first').copy()
after_review_id_dedup = len(df)

before_text_dedup = len(df)
duplicate_branch_text = df[df.duplicated(['official_name', 'clean_text'], keep=False)].copy()
df = df.drop_duplicates(['official_name', 'clean_text'], keep='first').copy()
after_branch_text_dedup = len(df)

empty_text = df[df['clean_text'].eq('')].copy()
df = df[~df['clean_text'].eq('')].copy()

for aspect, keywords in ASPECT_KEYWORDS.items():
    df[f'aspect_{aspect}'] = df['clean_text'].map(lambda text, kws=keywords: contains_keyword(text, kws))

aspect_columns = [f'aspect_{aspect}' for aspect in ASPECT_KEYWORDS]
df['aspect_count'] = df[aspect_columns].sum(axis=1)
df['aspect_list'] = df[aspect_columns].apply(
    lambda row: '|'.join(col.replace('aspect_', '') for col, has_aspect in row.items() if has_aspect),
    axis=1
)

df = df.sort_values(['official_name', 'review_rating_int', 'review_id']).reset_index(drop=True)
print('Input rows:', input_rows)
print('Output rows:', len(df))
display(df.head())

Input rows: 14785
Output rows: 14783


,review_id,official_name,official_province,google_name,review_rating,review_time_text,review_text,review_rating_int,clean_text,char_count,...,aspect_dokter,aspect_perawat,aspect_administrasi,aspect_farmasi,aspect_fasilitas,aspect_biaya_bpjs,aspect_kebersihan,aspect_komunikasi,aspect_count,aspect_list
0,049c6fe15de3dd92da21,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,1.0,3 bulan lalu,"Saya driver online, nganter ke dalam, terus da...",1,saya driver online nganter ke dalam terus dape...,207,...,False,False,False,False,True,False,False,False,1,fasilitas
1,04fdf8956382f2487006,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,1.0,Diedit 3 bulan lalu,Saya pasien rawat jalan. Tiap kontrol sistem s...,1,saya pasien rawat jalan tiap kontrol sistem se...,236,...,True,False,False,False,False,False,False,False,1,dokter
2,081e74ec426f3ebd553e,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,1.0,5 bulan lalu,"Pelayanannya payah, dari mulai satpam, perawat...",1,pelayanannya payah dari mulai satpam perawat b...,388,...,True,True,False,True,False,False,False,False,4,antrean|dokter|perawat|farmasi
3,0c5485730a126f94f8e8,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,1.0,2 bulan lalu,Sangat buruk nunggu lama.... Semua serba lama ...,1,sangat buruk nunggu lama semua serba lama lele...,154,...,False,False,False,False,False,False,False,False,1,antrean
4,12a49d7d04181cd5df78,Hermina Arcamanik,JAWA BARAT,RSU Hermina Arcamanik,1.0,2 bulan lalu,"kakak saya habis operasi, rawat inap di lantai...",1,kakak saya habis operasi rawat inap di lantai ...,232,...,False,False,False,False,True,False,False,False,1,fasilitas


## 5. Validasi Hasil Cleaning

In [5]:
validation = {
    'empty_clean_text': int(df['clean_text'].fillna('').eq('').sum()),
    'duplicate_review_id': int(df['review_id'].duplicated().sum()),
    'duplicate_branch_clean_text': int(df.duplicated(['official_name', 'clean_text']).sum()),
    'invalid_rating': int((~df['review_rating_int'].between(1, 5)).sum()),
    'branch_count': int(df['official_name'].nunique()),
    'province_count': int(df['official_province'].nunique()),
}
validation

{'empty_clean_text': 0,
 'duplicate_review_id': 0,
 'duplicate_branch_clean_text': 0,
 'invalid_rating': 0,
 'branch_count': 54,
 'province_count': 17}

## 6. Distribusi Sentimen dan Rating

In [6]:
sentiment_dist = df['sentiment_label'].value_counts().reindex(['negatif', 'netral', 'positif']).to_frame('count')
sentiment_dist['percentage'] = (sentiment_dist['count'] / len(df) * 100).round(2)
display(sentiment_dist)

rating_dist = df['review_rating_int'].value_counts().sort_index().to_frame('count')
rating_dist['percentage'] = (rating_dist['count'] / len(df) * 100).round(2)
display(rating_dist)

,count,percentage
sentiment_label,,
negatif,2241,15.16
netral,332,2.25
positif,12210,82.59


,count,percentage
review_rating_int,,
1,1816,12.28
2,425,2.87
3,332,2.25
4,440,2.98
5,11770,79.62


## 7. Simpan Dataset Bersih

In [7]:
interim_path = INTERIM_DIR / 'data_cleaning_interim.csv'
processed_path = PROCESSED_DIR / 'data_clean.csv'
model_ready_path = PROCESSED_DIR / 'data_model_ready.csv'

df.to_csv(interim_path, index=False, encoding='utf-8-sig')
df.to_csv(processed_path, index=False, encoding='utf-8-sig')
df[[
    'review_id', 'official_name', 'official_province', 'review_rating_int', 'review_time_text',
    'review_text', 'clean_text', 'sentiment_label', 'sentiment_id', 'word_count', 'aspect_list',
    *aspect_columns
]].to_csv(model_ready_path, index=False, encoding='utf-8-sig')

processed_path, model_ready_path

(WindowsPath('d:/Scraper adnan/preprocessing/data/processed/data_clean.csv'),
 WindowsPath('d:/Scraper adnan/preprocessing/data/processed/data_model_ready.csv'))

## 8. Buat Ringkasan Cabang untuk Sentiment Map

Ringkasan ini dipakai untuk peta sentimen, cabang terbaik, cabang prioritas, dan priority score.

In [8]:
branch_summary = (
    df.groupby(['official_name', 'official_province'], as_index=False)
    .agg(
        review_count=('review_id', 'count'),
        negatif_count=('sentiment_label', lambda s: int((s == 'negatif').sum())),
        netral_count=('sentiment_label', lambda s: int((s == 'netral').sum())),
        positif_count=('sentiment_label', lambda s: int((s == 'positif').sum())),
        mean_rating=('review_rating_int', 'mean'),
        mean_word_count=('word_count', 'mean'),
    )
)

for label in ['negatif', 'netral', 'positif']:
    branch_summary[f'{label}_pct'] = branch_summary.apply(
        lambda row, label=label: pct(int(row[f'{label}_count']), int(row['review_count'])), axis=1
    )

branch_summary['mean_rating'] = branch_summary['mean_rating'].round(2)
branch_summary['mean_word_count'] = branch_summary['mean_word_count'].round(2)
branch_summary['confidence_level'] = branch_summary['review_count'].map(confidence_from_count)
branch_summary['priority_score'] = (
    branch_summary['negatif_pct']
    + branch_summary['review_count'].clip(upper=300) / 300 * 10
    + branch_summary['netral_pct'] * 0.25
).round(2)

branch_summary = branch_summary.sort_values(['priority_score', 'negatif_pct'], ascending=False)
branch_summary.to_csv(REPORTS_DIR / 'sentiment_map_branch_ready.csv', index=False, encoding='utf-8-sig')
display(branch_summary.head(10))

,official_name,official_province,review_count,negatif_count,netral_count,positif_count,mean_rating,mean_word_count,negatif_pct,netral_pct,positif_pct,confidence_level,priority_score
8,Hermina Ciledug,JAWA BARAT,300,114,17,169,3.40,38.69,38.00,5.67,56.33,tinggi,49.42
5,Hermina Bitung,BANTEN,300,108,9,183,3.49,31.50,36.00,3.00,61.00,tinggi,46.75
49,Hermina Yogya,DI YOGYAKARTA,300,101,18,181,3.53,35.94,33.67,6.00,60.33,tinggi,45.17
2,Hermina Balikpapan,KALIMANTAN TIMUR,300,99,12,189,3.61,34.70,33.00,4.00,63.00,tinggi,44.00
10,Hermina Ciputat,BANTEN,300,99,12,189,3.62,39.79,33.00,4.00,63.00,tinggi,44.00
12,Hermina Daan Mogot,DKI JAKARTA,300,94,8,198,3.71,36.91,31.33,2.67,66.00,tinggi,42.00
41,Hermina Serpong,BANTEN,275,87,11,177,3.65,39.47,31.64,4.00,64.36,tinggi,41.81
9,Hermina Cilegon,BANTEN,300,83,13,204,3.82,28.53,27.67,4.33,68.00,tinggi,38.75
36,Hermina Periuk Tangerang,BANTEN,300,80,12,208,3.84,40.45,26.67,4.00,69.33,tinggi,37.67
24,Hermina Mekarsari,JAWA BARAT,300,73,8,219,3.96,35.74,24.33,2.67,73.00,tinggi,35.00


## 9. Buat Ringkasan Aspek Pelayanan

Ringkasan aspek dipakai untuk menjawab: positifnya tentang apa, negatifnya tentang apa, dan masalah dominan per cabang apa.

In [9]:
aspect_summary = []
for aspect, column in zip(ASPECT_KEYWORDS, aspect_columns):
    total_count = int(df[column].sum())
    neg_count = int(df.loc[df['sentiment_label'].eq('negatif'), column].sum())
    pos_count = int(df.loc[df['sentiment_label'].eq('positif'), column].sum())
    aspect_summary.append({
        'aspect': aspect,
        'total_count': total_count,
        'total_pct': pct(total_count, len(df)),
        'negative_count': neg_count,
        'negative_pct_of_negative_reviews': pct(neg_count, int(df['sentiment_label'].eq('negatif').sum())),
        'positive_count': pos_count,
        'positive_pct_of_positive_reviews': pct(pos_count, int(df['sentiment_label'].eq('positif').sum())),
    })

aspect_summary_df = pd.DataFrame(aspect_summary).sort_values('total_count', ascending=False)
aspect_summary_df.to_csv(REPORTS_DIR / 'aspect_summary.csv', index=False, encoding='utf-8-sig')
display(aspect_summary_df)

,aspect,total_count,total_pct,negative_count,negative_pct_of_negative_reviews,positive_count,positive_pct_of_positive_reviews
2,perawat,8546,57.81,512,22.85,7943,65.05
8,komunikasi,7913,53.53,656,29.27,7165,58.68
1,dokter,6940,46.95,891,39.76,5919,48.48
5,fasilitas,5399,36.52,600,26.77,4681,38.34
7,kebersihan,3807,25.75,155,6.92,3595,29.44
0,antrean,3751,25.37,1119,49.93,2465,20.19
4,farmasi,1601,10.83,552,24.63,968,7.93
6,biaya_bpjs,1574,10.65,651,29.05,855,7.00
3,administrasi,1477,9.99,477,21.29,898,7.35


## 10. Buat Ringkasan Aspek per Cabang

In [10]:
branch_aspect_rows = []
for (branch, province), group in df.groupby(['official_name', 'official_province']):
    negative_group = group[group['sentiment_label'].eq('negatif')]
    positive_group = group[group['sentiment_label'].eq('positif')]
    negative_aspects = {aspect: int(negative_group[f'aspect_{aspect}'].sum()) for aspect in ASPECT_KEYWORDS}
    positive_aspects = {aspect: int(positive_group[f'aspect_{aspect}'].sum()) for aspect in ASPECT_KEYWORDS}
    dominant_negative = max(negative_aspects, key=negative_aspects.get)
    dominant_positive = max(positive_aspects, key=positive_aspects.get)
    branch_aspect_rows.append({
        'official_name': branch,
        'official_province': province,
        'review_count': len(group),
        'negative_review_count': len(negative_group),
        'positive_review_count': len(positive_group),
        'dominant_negative_aspect': dominant_negative if negative_aspects[dominant_negative] > 0 else '',
        'dominant_negative_aspect_count': negative_aspects[dominant_negative],
        'dominant_positive_aspect': dominant_positive if positive_aspects[dominant_positive] > 0 else '',
        'dominant_positive_aspect_count': positive_aspects[dominant_positive],
    })

branch_aspect_summary = pd.DataFrame(branch_aspect_rows).sort_values(
    ['negative_review_count', 'dominant_negative_aspect_count'], ascending=False
)
branch_aspect_summary.to_csv(REPORTS_DIR / 'branch_aspect_summary.csv', index=False, encoding='utf-8-sig')
display(branch_aspect_summary.head(15))

,official_name,official_province,review_count,negative_review_count,positive_review_count,dominant_negative_aspect,dominant_negative_aspect_count,dominant_positive_aspect,dominant_positive_aspect_count
8,Hermina Ciledug,JAWA BARAT,300,114,169,antrean,58,komunikasi,101
5,Hermina Bitung,BANTEN,300,108,183,antrean,49,perawat,109
49,Hermina Yogya,DI YOGYAKARTA,300,101,181,antrean,52,dokter,100
10,Hermina Ciputat,BANTEN,300,99,189,antrean,68,perawat,121
2,Hermina Balikpapan,KALIMANTAN TIMUR,300,99,189,antrean,50,perawat,106
12,Hermina Daan Mogot,DKI JAKARTA,300,94,198,antrean,49,perawat,123
41,Hermina Serpong,BANTEN,275,87,177,antrean,45,perawat,132
9,Hermina Cilegon,BANTEN,300,83,204,antrean,49,komunikasi,129
36,Hermina Periuk Tangerang,BANTEN,300,80,208,dokter,33,perawat,140
24,Hermina Mekarsari,JAWA BARAT,300,73,219,antrean,40,perawat,128


## 11. Simpan Laporan Distribusi dan Summary

In [11]:
sentiment_dist.reset_index(names='sentiment_label').to_csv(REPORTS_DIR / 'sentiment_distribution.csv', index=False, encoding='utf-8-sig')
rating_dist.reset_index(names='review_rating_int').to_csv(REPORTS_DIR / 'rating_distribution.csv', index=False, encoding='utf-8-sig')

invalid_rating.to_csv(REPORTS_DIR / 'removed_rows' / 'invalid_rating_rows.csv', index=False, encoding='utf-8-sig')
duplicate_review_id.to_csv(REPORTS_DIR / 'removed_rows' / 'duplicate_review_id_rows.csv', index=False, encoding='utf-8-sig')
duplicate_branch_text.to_csv(REPORTS_DIR / 'removed_rows' / 'duplicate_branch_clean_text_rows.csv', index=False, encoding='utf-8-sig')
empty_text.to_csv(REPORTS_DIR / 'removed_rows' / 'empty_clean_text_rows.csv', index=False, encoding='utf-8-sig')

summary = {
    'input_file': str(INPUT_PATH),
    'input_rows': input_rows,
    'output_rows': len(df),
    'removed_rows_total': input_rows - len(df),
    'invalid_rating_rows': len(invalid_rating),
    'detected_duplicate_review_id_rows': len(duplicate_review_id),
    'detected_duplicate_branch_clean_text_rows': len(duplicate_branch_text),
    'removed_by_review_id_dedup': before_review_id_dedup - after_review_id_dedup,
    'removed_by_branch_text_dedup': before_text_dedup - after_branch_text_dedup,
    'empty_clean_text_rows': len(empty_text),
    'branch_count': int(df['official_name'].nunique()),
    'province_count': int(df['official_province'].nunique()),
    'sentiment_distribution': df['sentiment_label'].value_counts().reindex(['negatif', 'netral', 'positif'], fill_value=0).to_dict(),
    'rating_distribution': {str(k): int(v) for k, v in df['review_rating_int'].value_counts().sort_index().items()},
    'outputs': {
        'processed_data': str(processed_path),
        'model_ready_data': str(model_ready_path),
        'branch_sentiment_map': str(REPORTS_DIR / 'sentiment_map_branch_ready.csv'),
        'aspect_summary': str(REPORTS_DIR / 'aspect_summary.csv'),
        'branch_aspect_summary': str(REPORTS_DIR / 'branch_aspect_summary.csv'),
    }
}

(REPORTS_DIR / 'cleaning_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
summary

{'input_file': 'd:\\Scraper adnan\\preprocessing\\data_fix.csv',
 'input_rows': 14785,
 'output_rows': 14783,
 'removed_rows_total': 2,
 'invalid_rating_rows': 0,
 'detected_duplicate_review_id_rows': 0,
 'detected_duplicate_branch_clean_text_rows': 4,
 'removed_by_review_id_dedup': 0,
 'removed_by_branch_text_dedup': 2,
 'empty_clean_text_rows': 0,
 'branch_count': 54,
 'province_count': 17,
 'sentiment_distribution': {'negatif': 2241, 'netral': 332, 'positif': 12210},
 'rating_distribution': {'1': 1816, '2': 425, '3': 332, '4': 440, '5': 11770},
 'outputs': {'processed_data': 'd:\\Scraper adnan\\preprocessing\\data\\processed\\data_clean.csv',
  'model_ready_data': 'd:\\Scraper adnan\\preprocessing\\data\\processed\\data_model_ready.csv',
  'branch_sentiment_map': 'd:\\Scraper adnan\\preprocessing\\reports\\sentiment_map_branch_ready.csv',
  'aspect_summary': 'd:\\Scraper adnan\\preprocessing\\reports\\aspect_summary.csv',
  'branch_aspect_summary': 'd:\\Scraper adnan\\preprocessing\

## 12. Kesimpulan Cleaning

Output tahap cleaning sudah siap untuk tahap berikutnya:
- EDA lanjutan,
- validasi label sentimen,
- modeling deep learning,
- sentiment map,
- detail cabang berdasarkan aspek pelayanan.